# PS-48 — Distracted Driver Detection
**CBL — Deep Learning | Image Classification Challenge**

| | |
|---|---|
| **Name** | Lokaraj R |
| **Register No.** | RA2311026050005 |
| **Section** | AIML B |
| **Problem statement** | PS-48 Distracted Driver Detection (Human-Centred Vision) |
| **Dataset** | State Farm Distracted Driver Detection (Kaggle competition), 22,424 labelled images, 26 drivers, 10 classes |

**Objective.** Classify a dashboard-camera image of a driver into one of ten behaviours (c0 safe driving … c9 talking to passenger).

**Specific challenge: data leakage through the driver.** Each driver was filmed in one continuous session, so the frames of one driver are near-duplicates. If the same driver is in both train and test (a random split), the model can memorise *who the driver is, their clothes and their car* and still score close to 100 %. That number says nothing about a **new** driver. All results here therefore use a **driver-grouped split** (no driver appears in more than one of train/val/test). A random split is trained as well, as a controlled experiment that measures how much a random split overstates the result.

**Success metric.** **Macro F1 on unseen test drivers** (all ten behaviours count equally; a safety system must not ignore a rare behaviour). Also reported: accuracy, per-class precision/recall/F1, confusion matrix, top-2 accuracy and multi-class **log loss** (the official Kaggle metric for this competition).

**Notebook map.** 0 Setup & reproducibility → 1 Data & EDA → 2 Balanced subset & driver-grouped split → 3 Leakage evidence (near-duplicates) → 4 Pre-processing & augmentation → 5 Models → 6 Training → 7 Model selection on validation → 8 One-time test evaluation → 9 Error analysis & Grad-CAM → 10 Robustness test → 11 Random split vs driver split → 12 Summary.

## 0. Setup and reproducibility
Everything that affects the numbers is fixed here: random seed, subset size, split sizes and hyper-parameters. Library versions are printed and saved to `outputs/versions.json`, and the exact splits are saved to `outputs/splits/`.

**Getting the data.** Log in to Kaggle, open <https://www.kaggle.com/c/state-farm-distracted-driver-detection>, accept the competition rules, download the data and unzip it so that this folder contains `data/driver_imgs_list.csv` and `data/imgs/train/c0 … c9/`. On a Kaggle notebook, add the competition as input and the path is detected automatically. **On Google Colab**, select *Runtime → Change runtime type → T4 GPU*, add your Kaggle API token as a Colab secret named `KAGGLE_API_TOKEN`, and the next cell downloads the data.

In [ ]:
import os, json, time, copy, random, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
import torchvision.transforms.functional as TF

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, log_loss, top_k_accuracy_score,
                             classification_report, confusion_matrix,
                             precision_recall_fscore_support)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 9})

In [ ]:
# Google Colab only: download the competition data with kagglehub.
# Add your Kaggle API token as a Colab secret named KAGGLE_API_TOKEN (key icon in the left bar), or log in when asked.
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub"], check=True)
    import kagglehub
    try:
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        kagglehub.login()
    path = kagglehub.competition_download("state-farm-distracted-driver-detection")
    os.environ["PS48_DATA_DIR"] = str(next(Path(path).rglob("driver_imgs_list.csv")).parent)
    print("data downloaded to", os.environ["PS48_DATA_DIR"])

In [ ]:
# ---------------- configuration (everything that changes the numbers) ----------------
SEED = 42
QUICK = os.environ.get("PS48_QUICK") == "1"     # smoke test only: tiny run to check the pipeline end to end

PER_DRIVER_CLASS = 16      # images sampled per (driver, class) cell -> ~4,100-image balanced subset
                           # (Part G: ~300-500 images per class). Set to None to use all 22,424 images.
N_TEST_DRIVERS = 5         # drivers held out completely for the test set
N_VAL_DRIVERS = 4          # drivers held out for validation / early stopping / model selection

IMG_SIZE = 224             # ImageNet input size of the pretrained backbones
CACHE_W, CACHE_H = 320, 240  # images are resized once (640x480 -> 320x240, same 4:3 shape) to speed up loading
BATCH_SIZE = 32

EPOCHS_SCRATCH = 20        # CNN trained from scratch
EPOCHS_HEAD = 2            # transfer learning, stage 1: frozen backbone, train the new head only
EPOCHS_FT = 10             # transfer learning, stage 2: unfreeze and fine-tune the whole network
PATIENCE = 4               # early stopping on validation macro F1 (counter restarts at each stage)
LR_SCRATCH, LR_HEAD, LR_FT = 1e-3, 1e-3, 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

MODEL_NAMES = ["scratch_cnn", "mobilenet_v3_large", "resnet18", "efficientnet_b0"]

if QUICK:
    PER_DRIVER_CLASS, EPOCHS_SCRATCH, EPOCHS_HEAD, EPOCHS_FT = 2, 1, 1, 1

OUT = Path("outputs")
FIG, SPLITS, MODELS_DIR = OUT / "figures", OUT / "splits", OUT / "models"
for d in (FIG, SPLITS, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLASSES = [f"c{i}" for i in range(10)]
CLASS_NAMES = {"c0": "safe driving", "c1": "texting - right", "c2": "phone - right",
               "c3": "texting - left", "c4": "phone - left", "c5": "operating radio",
               "c6": "drinking", "c7": "reaching behind", "c8": "hair & makeup",
               "c9": "talking to passenger"}
NUM_CLASSES = len(CLASSES)
TICKS = [f"{c}\n{CLASS_NAMES[c]}" for c in CLASSES]

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
# worker processes need fork (Linux); on macOS/Windows notebooks load in the main process
NUM_WORKERS = 2 if platform.system() == "Linux" else 0

versions = {"python": platform.python_version(), "platform": platform.platform(),
            "device": str(DEVICE), "torch": torch.__version__,
            "torchvision": torchvision.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
            "matplotlib": matplotlib.__version__, "seed": SEED, "quick_mode": QUICK}
(OUT / "versions.json").write_text(json.dumps(versions, indent=2))
pd.Series(versions)

## 1. Data loading and exploratory analysis
`driver_imgs_list.csv` gives, for every training image, the driver (`subject`), the class folder and the file name. The driver id is what the split must be grouped by.

In [ ]:
def find_data_dir():
    candidates = [Path(os.environ["PS48_DATA_DIR"])] if os.environ.get("PS48_DATA_DIR") else []
    candidates += [Path("data"), Path("data/state-farm-distracted-driver-detection"),
                   Path("/kaggle/input/state-farm-distracted-driver-detection"), Path("/content/data")]
    for c in candidates:
        if (c / "driver_imgs_list.csv").exists():
            return c
    raise FileNotFoundError("driver_imgs_list.csv not found - see 'Getting the data' above.")

DATA_DIR = find_data_dir()
IMG_DIR = DATA_DIR / "imgs" / "train" if (DATA_DIR / "imgs" / "train").exists() else DATA_DIR / "train"

df = pd.read_csv(DATA_DIR / "driver_imgs_list.csv")
df["label"] = df["classname"].str[1:].astype(int)
df["path"] = [str(IMG_DIR / c / f) for c, f in zip(df["classname"], df["img"])]
missing = sum(not Path(p).exists() for p in df["path"])

print(f"data folder      : {DATA_DIR}")
print(f"labelled images  : {len(df):,}   (missing files: {missing})")
print(f"drivers          : {df['subject'].nunique()}")
print(f"image size       : {Image.open(df['path'].iloc[0]).size} (W x H)")
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [1, 1.3]})
cc = df["classname"].value_counts().reindex(CLASSES)
ax[0].bar(range(NUM_CLASSES), cc.values, color="#4C72B0")
ax[0].set_xticks(range(NUM_CLASSES), TICKS, rotation=90)
ax[0].set_ylabel("images"); ax[0].set_title(f"Images per class (imbalance ratio {cc.max() / cc.min():.2f})")
dc = df["subject"].value_counts().sort_index()
ax[1].bar(dc.index, dc.values, color="#55A868")
ax[1].tick_params(axis="x", rotation=90)
ax[1].set_ylabel("images"); ax[1].set_title("Images per driver")
plt.savefig(FIG / "eda_class_driver_counts.png"); plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(pd.crosstab(df["subject"], df["classname"]), annot=True, fmt="d", cmap="Blues",
            annot_kws={"size": 7}, cbar_kws={"label": "images"})
plt.title("Images per driver and class - every driver performs every behaviour")
plt.xlabel("class"); plt.ylabel("driver")
plt.savefig(FIG / "eda_driver_class_heatmap.png"); plt.show()

print(cc.rename(CLASS_NAMES).to_string())

In [ ]:
# four samples per class, each from a different driver
rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(4, NUM_CLASSES, figsize=(20, 7))
for j, c in enumerate(CLASSES):
    rows = df[df["classname"] == c]
    drivers = rng.choice(rows["subject"].unique(), 4, replace=False)
    for i, d in enumerate(drivers):
        r = rows[rows["subject"] == d].sample(1, random_state=SEED).iloc[0]
        axes[i, j].imshow(Image.open(r["path"]))
        axes[i, j].set_title(f"{c} {CLASS_NAMES[c]}\n{d}" if i == 0 else d, fontsize=8)
        axes[i, j].axis("off")
plt.suptitle("Sample images per class (column) from different drivers (rows)")
plt.tight_layout(); plt.savefig(FIG / "eda_samples.png"); plt.show()

**EDA observations.**
* The ten classes are close to balanced (largest/smallest ratio printed above), so class imbalance is a minor issue; the balanced subset below removes it entirely.
* Every one of the 26 drivers performs all ten behaviours, but drivers contribute very different numbers of images.
* The images are frames from a video of each driver, so frames of the same driver/class are almost identical. **This is the main dataset risk (leakage):** a random split puts near-copies of a test frame into the training set.
* Left/right matters: c1/c3 (texting right/left) and c2/c4 (phone right/left) differ only by which hand is used. **Horizontal flip is therefore not used as augmentation** — it would turn a c1 image into something that looks like c3 while keeping the c1 label.

## 2. Balanced subset and driver-grouped split
Following the lab note for large datasets, a fixed balanced subset is drawn: the same number of images from every (driver, class) cell, so the subset is balanced over classes **and** drivers. Drivers (not images) are then shuffled with the fixed seed and assigned to train / validation / test.

In [ ]:
if PER_DRIVER_CLASS:
    sub = (df.sample(frac=1.0, random_state=SEED)
             .groupby(["subject", "classname"]).head(PER_DRIVER_CLASS))
else:
    sub = df.copy()
sub = sub.sort_values(["subject", "classname", "img"]).reset_index(drop=True)

drivers = np.array(sorted(sub["subject"].unique()))
perm = np.random.RandomState(SEED).permutation(drivers)
test_drivers = sorted(perm[:N_TEST_DRIVERS])
val_drivers = sorted(perm[N_TEST_DRIVERS:N_TEST_DRIVERS + N_VAL_DRIVERS])
train_drivers = sorted(perm[N_TEST_DRIVERS + N_VAL_DRIVERS:])

sub["split"] = "train"
sub.loc[sub["subject"].isin(val_drivers), "split"] = "val"
sub.loc[sub["subject"].isin(test_drivers), "split"] = "test"

# the PS-48 rule: each driver in exactly one split
groups = {s: set(sub.loc[sub["split"] == s, "subject"]) for s in ["train", "val", "test"]}
assert not (groups["train"] & groups["val"]) and not (groups["train"] & groups["test"]) \
       and not (groups["val"] & groups["test"]), "a driver appears in more than one split"
print("driver overlap between splits: none")

print(f"subset: {len(sub):,} images ({PER_DRIVER_CLASS} per driver-class cell)")
for s, d in [("train", train_drivers), ("val", val_drivers), ("test", test_drivers)]:
    print(f"{s:5s}: {len(d):2d} drivers, {(sub['split'] == s).sum():5d} images  {', '.join(d)}")
sub[["img", "classname", "subject", "split"]].to_csv(SPLITS / "driver_split.csv", index=False)
pd.crosstab(sub["split"], sub["classname"]).reindex(["train", "val", "test"])

**Control: random split (for the leakage experiment only).** Same subset and the same split sizes, but images are assigned at random (stratified by class), ignoring the driver. This is the "wrong" split the problem statement warns about; it is used only in Section 11.

In [ ]:
rand = sub.copy()
n_test, n_val = (sub["split"] == "test").sum(), (sub["split"] == "val").sum()
trval_idx, te_idx = train_test_split(rand.index, test_size=n_test, stratify=rand["label"], random_state=SEED)
tr_idx, va_idx = train_test_split(trval_idx, test_size=n_val, stratify=rand.loc[trval_idx, "label"],
                                  random_state=SEED)
rand["split"] = "train"
rand.loc[va_idx, "split"] = "val"
rand.loc[te_idx, "split"] = "test"
rand[["img", "classname", "subject", "split"]].to_csv(SPLITS / "random_split.csv", index=False)

seen = set(rand.loc[rand["split"] == "test", "subject"]) & set(rand.loc[rand["split"] == "train", "subject"])
print(f"random split: {len(seen)} of {rand.loc[rand['split'] == 'test', 'subject'].nunique()} "
      f"test drivers also appear in training")

## 3. Evidence of leakage: near-duplicate frames
Before any training, measure how similar each test image is to its **closest training image** (root-mean-square pixel difference of 32×24 grayscale thumbnails, 0 = identical). Images are first resized once to 320×240 and cached on disk, which also makes training faster.

In [ ]:
CACHE = Path("cache") / f"{CACHE_W}x{CACHE_H}"

def cache_path(row):
    return str(CACHE / row.classname / row.img)

def make_cached(src_dst):
    src, dst = src_dst
    if not Path(dst).exists():
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        Image.open(src).convert("RGB").resize((CACHE_W, CACHE_H), Image.BILINEAR).save(dst, quality=95)

sub["cpath"] = [cache_path(r) for r in sub.itertuples()]
rand["cpath"] = sub["cpath"]
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    list(ex.map(make_cached, zip(sub["path"], sub["cpath"])))
print(f"cached {len(sub):,} images in {time.time() - t0:.0f}s")

thumbs = np.stack([np.asarray(Image.open(p).convert("L").resize((32, 24)), np.float32).ravel() / 255.
                   for p in sub["cpath"]])

def nearest_train(split):
    tr, te = np.where(split == "train")[0], np.where(split == "test")[0]
    d = torch.cdist(torch.from_numpy(thumbs[te]), torch.from_numpy(thumbs[tr])) / np.sqrt(thumbs.shape[1])
    dist, arg = d.min(1)
    return te, tr[arg.numpy()], dist.numpy()

nn_driver = nearest_train(sub["split"].values)
nn_random = nearest_train(rand["split"].values)
near_dup_thr = 0.05
nn_stats = {name: {"median_rms": float(np.median(r[2])), "share_below_0.05": float((r[2] < near_dup_thr).mean())}
            for name, r in [("driver_split", nn_driver), ("random_split", nn_random)]}
print(json.dumps(nn_stats, indent=2))

plt.figure(figsize=(7, 3.4))
bins = np.linspace(0, max(nn_driver[2].max(), nn_random[2].max()), 50)
plt.hist(nn_random[2], bins, alpha=.7, label="random split", color="#C44E52")
plt.hist(nn_driver[2], bins, alpha=.7, label="driver-grouped split", color="#4C72B0")
plt.xlabel("RMS pixel distance of a test image to its nearest training image")
plt.ylabel("test images"); plt.legend(); plt.title("How close is each test image to the training set?")
plt.savefig(FIG / "leakage_nn_hist.png"); plt.show()

# three examples: a random-split test image and its nearest training image
te, tr, dist = nn_random
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
for k, i in enumerate(np.argsort(dist)[[0, len(dist) // 4, len(dist) // 2]]):
    for row, idx, lab in [(0, te[i], "TEST"), (1, tr[i], "TRAIN (nearest)")]:
        r = sub.iloc[idx]
        axes[row, k].imshow(Image.open(r["cpath"])); axes[row, k].axis("off")
        axes[row, k].set_title(f"{lab}: {r['subject']} {r['classname']}" + (f"\nRMS={dist[i]:.3f}" if row == 0 else ""), fontsize=8)
plt.suptitle("Random split: test images have a near-copy of the same driver in training")
plt.tight_layout(); plt.savefig(FIG / "leakage_examples.png"); plt.show()

## 4. Pre-processing and augmentation
* **Resize** the cached 320×240 image to 224×224 and **normalise** with the ImageNet mean/std (required by the pretrained backbones).
* **Training-only augmentation**, chosen to mimic changes between cars and drivers rather than to create new behaviours: random resized crop (camera position / zoom), small rotation (camera mounting), colour jitter (lighting, skin tone, clothing colour) and random erasing (partial occlusion, and it stops the model from relying on one small patch).
* **No horizontal flip** (it swaps left/right classes, see EDA). **No augmentation on validation or test.**

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(1.1, 1.5)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class DriverDataset(Dataset):
    def __init__(self, frame, tfm):
        self.paths, self.labels, self.tfm = frame["cpath"].tolist(), frame["label"].tolist(), tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tfm(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

def make_loaders(frame):
    loaders = {}
    for s in ["train", "val", "test"]:
        is_train = s == "train"
        loaders[s] = DataLoader(DriverDataset(frame[frame["split"] == s], train_tfms if is_train else eval_tfms),
                                batch_size=BATCH_SIZE, shuffle=is_train, drop_last=is_train,
                                generator=torch.Generator().manual_seed(SEED) if is_train else None,
                                num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    return loaders

def denorm(x):
    return (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0).numpy()

loaders_driver = make_loaders(sub)
print({s: len(l.dataset) for s, l in loaders_driver.items()})

ex = Image.open(sub["cpath"].iloc[0]).convert("RGB")
fig, axes = plt.subplots(1, 6, figsize=(16, 2.6))
axes[0].imshow(ex.resize((IMG_SIZE, IMG_SIZE))); axes[0].set_title("original (resized)")
for a in axes[1:]:
    a.imshow(denorm(train_tfms(ex))); a.set_title("augmented")
for a in axes: a.axis("off")
plt.savefig(FIG / "augmentation_examples.png"); plt.show()

## 5. Models
| Model | Type | Why |
|---|---|---|
| **Scratch CNN** | 5 conv blocks (2× conv-BN-ReLU + max-pool), global average pooling, dropout, linear | Baseline: how far does a CNN get without prior knowledge? |
| **MobileNetV3-Large** | ImageNet-pretrained, transfer learning | Lightweight model that could run on an in-car device |
| **ResNet-18** | ImageNet-pretrained, transfer learning | Standard residual backbone, mid-size |
| **EfficientNet-B0** | ImageNet-pretrained, transfer learning | Strong accuracy per parameter |

**Transfer-learning recipe (two stages).** Replace the 1000-class ImageNet layer with a new 10-class layer. Stage 1: freeze the backbone and train only the new head (`LR_HEAD`), so the random head does not destroy pretrained features with large gradients. Stage 2: unfreeze everything and fine-tune with a lower learning rate (`LR_FT`).

**Common settings.** Loss: cross-entropy with label smoothing 0.1 (frames are ambiguous at the start and end of an action; smoothing reduces over-confidence). Optimiser: AdamW with weight decay, one-cycle learning-rate schedule. Early stopping on **validation macro F1** with patience `PATIENCE`; the best epoch's weights are restored.

In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

def build_model(name):
    # returns (model, new head to train in stage 1, last conv block for Grad-CAM)
    if name == "scratch_cnn":
        m = ScratchCNN()
        return m, m.head, m.features[-1]
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, NUM_CLASSES))
        return m, m.fc, m.layer4
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
        return m, m.classifier, m.features[-1]
    if name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
        return m, m.classifier, m.features[-1]
    raise ValueError(name)

def stages_for(name):
    # (stage name, epochs, learning rate, freeze backbone?)
    if name == "scratch_cnn":
        return [("scratch", EPOCHS_SCRATCH, LR_SCRATCH, False)]
    return [("head", EPOCHS_HEAD, LR_HEAD, True), ("finetune", EPOCHS_FT, LR_FT, False)]

def n_params(m):
    return sum(p.numel() for p in m.parameters())

for name in MODEL_NAMES:
    print(f"{name:20s} {n_params(build_model(name)[0]) / 1e6:6.2f} M parameters")

## 6. Training

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        probs.append(F.softmax(model(x.to(DEVICE)), 1).cpu())
        ys.append(y)
    p = torch.cat(probs).double().numpy()
    return p / p.sum(1, keepdims=True), torch.cat(ys).numpy()

def compute_metrics(y, p):
    pred = p.argmax(1)
    labels = list(range(NUM_CLASSES))
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "log_loss": log_loss(y, p, labels=labels), "top2_acc": top_k_accuracy_score(y, p, k=2, labels=labels)}

def train_one_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    tot_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()
        tot_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return tot_loss / n, correct / n

def fit(name, loaders, tag):
    seed_everything()
    loaders["train"].generator.manual_seed(SEED)    # same batch order for every model
    model, head, _ = build_model(name)
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history, best_f1, best_state, best_epoch, epoch = [], -1.0, None, 0, 0
    t0 = time.time()
    for stage, epochs, lr, freeze in stages_for(name):
        for p in model.parameters():
            p.requires_grad = not freeze
        for p in head.parameters():
            p.requires_grad = True
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, pct_start=0.2,
                                                        total_steps=epochs * len(loaders["train"]))
        wait = 0
        for _ in range(epochs):
            epoch += 1
            tr_loss, tr_acc = train_one_epoch(model, loaders["train"], criterion, optimizer, scheduler)
            p, y = predict(model, loaders["val"])
            m = compute_metrics(y, p)
            history.append({"epoch": epoch, "stage": stage, "train_loss": tr_loss, "train_acc": tr_acc,
                            "val_loss": m["log_loss"], "val_acc": m["accuracy"], "val_f1": m["macro_f1"]})
            print(f"[{name}|{tag}] ep {epoch:2d} {stage:8s} train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
                  f"val loss {m['log_loss']:.3f} acc {m['accuracy']:.3f} F1 {m['macro_f1']:.3f} "
                  f"| {time.time() - t0:5.0f}s")
            if m["macro_f1"] > best_f1:
                best_f1, best_epoch, wait = m["macro_f1"], epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f"  early stopping in stage '{stage}' (no val F1 gain for {PATIENCE} epochs)")
                    break
    model.load_state_dict(best_state)
    info = {"best_epoch": best_epoch, "epochs_run": epoch, "best_val_f1": best_f1,
            "train_time_min": (time.time() - t0) / 60, "params_M": n_params(model) / 1e6}
    torch.save(model.state_dict(), MODELS_DIR / f"{name}_{tag}.pt")
    pd.DataFrame(history).to_csv(OUT / f"history_{name}_{tag}.csv", index=False)
    print(f"  -> best epoch {best_epoch}, val macro F1 {best_f1:.3f}, {info['train_time_min']:.1f} min")
    return model, pd.DataFrame(history), info

def plot_history(h, title, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].plot(h["epoch"], h["train_loss"], "o-", ms=3, label="train (label-smoothed CE)")
    ax[0].plot(h["epoch"], h["val_loss"], "o-", ms=3, label="validation (CE = log loss)")
    ax[0].set_ylabel("loss")
    ax[1].plot(h["epoch"], h["train_acc"], "o-", ms=3, label="train accuracy (augmented)")
    ax[1].plot(h["epoch"], h["val_acc"], "o-", ms=3, label="validation accuracy")
    ax[1].plot(h["epoch"], h["val_f1"], "s--", ms=3, label="validation macro F1")
    ax[1].set_ylabel("accuracy / F1")
    change = h.index[h["stage"] != h["stage"].shift()][1:]
    for a in ax:
        for c in change:
            a.axvline(h.loc[c, "epoch"] - 0.5, color="grey", ls=":", lw=1)
        a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend(fontsize=8)
    fig.suptitle(title)
    plt.savefig(path); plt.show()

In [ ]:
runs = {}
for name in MODEL_NAMES:
    model, hist, info = fit(name, loaders_driver, tag="driver")
    runs[name] = {"model": model, "history": hist, "info": info}
    plot_history(hist, f"{name} - driver-grouped split (dotted line: backbone unfrozen)",
                 FIG / f"curves_{name}.png")

## 7. Model selection (validation drivers only)
The model is chosen on the **validation** drivers. The test drivers have not been used for any decision so far.

In [ ]:
def cpu_latency_ms(model, n=30):
    m = copy.deepcopy(model).cpu().eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        for _ in range(5):
            m(x)
        t0 = time.perf_counter()
        for _ in range(n):
            m(x)
    return (time.perf_counter() - t0) / n * 1000

val_table = pd.DataFrame({name: {"best val macro F1": r["info"]["best_val_f1"],
                                 "best epoch": r["info"]["best_epoch"],
                                 "params (M)": r["info"]["params_M"],
                                 "CPU latency (ms/img)": cpu_latency_ms(r["model"]),
                                 "train time (min)": r["info"]["train_time_min"]}
                          for name, r in runs.items()}).T
BEST = val_table["best val macro F1"].idxmax()
print(f"selected on validation: {BEST}")
val_table.round(3)

## 8. Test evaluation (run once)
All models are evaluated once on the five unseen test drivers. Only the model selected in Section 7 is analysed in detail.

In [ ]:
test_preds, rows = {}, []
for name, r in runs.items():
    p, y = predict(r["model"], loaders_driver["test"])
    test_preds[name] = (p, y)
    m = compute_metrics(y, p)
    rows.append({"model": name, "test accuracy": m["accuracy"], "test macro F1": m["macro_f1"],
                 "test log loss": m["log_loss"], "test top-2 acc": m["top2_acc"],
                 "params (M)": val_table.loc[name, "params (M)"],
                 "CPU latency (ms/img)": val_table.loc[name, "CPU latency (ms/img)"],
                 "train time (min)": val_table.loc[name, "train time (min)"]})
test_table = pd.DataFrame(rows).set_index("model")
test_table.to_csv(OUT / "test_results_all_models.csv")
test_table.round(3)

In [ ]:
p_best, y_test = test_preds[BEST]
pred_best = p_best.argmax(1)
print(f"{BEST} - per-class results on the test drivers\n")
print(classification_report(y_test, pred_best, labels=list(range(NUM_CLASSES)), zero_division=0, target_names=[f"{c} {CLASS_NAMES[c]}" for c in CLASSES], digits=3))
prf = pd.DataFrame(precision_recall_fscore_support(y_test, pred_best, labels=list(range(NUM_CLASSES))),
                   index=["precision", "recall", "f1", "support"], columns=CLASSES).T

cm = confusion_matrix(y_test, pred_best, labels=list(range(NUM_CLASSES)))
cm_pct = cm / cm.sum(1, keepdims=True) * 100
plt.figure(figsize=(8.5, 7))
sns.heatmap(cm_pct, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=100, xticklabels=TICKS, yticklabels=TICKS,
            cbar_kws={"label": "% of true class (colour); numbers = image counts"})
plt.xlabel("predicted class"); plt.ylabel("true class")
plt.title(f"Confusion matrix - {BEST}, test drivers ({', '.join(test_drivers)})")
plt.savefig(FIG / "confusion_matrix_best.png"); plt.show()

off = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j]
confused = sorted(off, reverse=True)[:5]
print("most confused pairs (true -> predicted):")
for n, a, b in confused:
    print(f"  {a} {CLASS_NAMES[a]:20s} -> {b} {CLASS_NAMES[b]:20s}: {n} images")

In [ ]:
# does performance depend on who is driving?
test_df = sub[sub["split"] == "test"].reset_index(drop=True)
test_df["pred"] = pred_best
test_df["conf"] = p_best.max(1)
per_driver = test_df.assign(correct=test_df["pred"] == test_df["label"]).groupby("subject")["correct"].mean()
plt.figure(figsize=(6, 2.8))
plt.bar(per_driver.index, per_driver.values, color="#8172B2")
plt.axhline(per_driver.mean(), color="k", ls="--", lw=1, label=f"mean {per_driver.mean():.2f}")
plt.ylim(0, 1); plt.ylabel("accuracy"); plt.title(f"{BEST}: accuracy per unseen test driver"); plt.legend()
plt.savefig(FIG / "per_driver_accuracy.png"); plt.show()
per_driver.round(3)

## 9. Error analysis and Grad-CAM
Grad-CAM highlights the image regions that most increased the score of the predicted class. A sensible model should look at the hands, phone, face and radio area rather than at the background or the driver's clothes.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.hook = layer.register_forward_hook(self._save)
    def _save(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))
    def __call__(self, x, cls=None):
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(True)
        out = self.model(x.unsqueeze(0).to(DEVICE))
        cls = int(out.argmax(1)) if cls is None else cls
        self.model.zero_grad()
        out[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
        return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), cls
    def remove(self):
        self.hook.remove()

best_model = runs[BEST]["model"]
# locate the Grad-CAM layer inside the trained model (same attribute path as in build_model)
cam_layer = {"scratch_cnn": lambda m: m.features[-1], "resnet18": lambda m: m.layer4,
             "efficientnet_b0": lambda m: m.features[-1], "mobilenet_v3_large": lambda m: m.features[-1]}[BEST](best_model)
cam = GradCAM(best_model, cam_layer)

def show_cam(ax, row, title):
    img = Image.open(row["cpath"]).convert("RGB")
    heat, _ = cam(eval_tfms(img), int(row["pred"]))
    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE)))
    ax.imshow(heat, cmap="jet", alpha=0.4)
    ax.set_title(title, fontsize=8); ax.axis("off")

In [ ]:
wrong = test_df[test_df["pred"] != test_df["label"]].sort_values("conf", ascending=False)
# up to 8 errors: the most confident mistake of each (true, predicted) pair first, so different error types are shown
examples = wrong.drop_duplicates(["label", "pred"]).head(8)
examples = pd.concat([examples, wrong.drop(examples.index)]).head(8)
n = len(examples)
fig, axes = plt.subplots(int(np.ceil(n / 2)), 4, figsize=(13, 3.3 * np.ceil(n / 2)), squeeze=False)
for k, (_, r) in enumerate(examples.iterrows()):
    a1, a2 = axes[k // 2, (k % 2) * 2], axes[k // 2, (k % 2) * 2 + 1]
    a1.imshow(Image.open(r["cpath"])); a1.axis("off")
    a1.set_title(f"#{k + 1} {r['subject']}  true {r['classname']} {CLASS_NAMES[r['classname']]}\n"
                 f"pred c{r['pred']} {CLASS_NAMES['c' + str(r['pred'])]} ({r['conf']:.2f})", fontsize=8)
    show_cam(a2, r, "Grad-CAM for the predicted class")
for a in axes.ravel()[2 * n:]:
    a.axis("off")
plt.suptitle(f"Misclassified test images - {BEST}", y=1.0)
plt.tight_layout(); plt.savefig(FIG / "misclassified_gradcam.png"); plt.show()
examples[["img", "subject", "classname", "pred", "conf"]].assign(
    predicted=lambda d: "c" + d["pred"].astype(str)).drop(columns="pred").reset_index(drop=True)

In [ ]:
# Grad-CAM on the most confident correct prediction of each class
right = test_df[test_df["pred"] == test_df["label"]].sort_values("conf", ascending=False).drop_duplicates("label")
right = right.set_index("label").reindex(range(NUM_CLASSES)).dropna(subset=["cpath"]).reset_index()
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for a in axes.ravel():
    a.axis("off")
for a, (_, r) in zip(axes.ravel(), right.iterrows()):
    r["pred"] = int(r["label"])
    show_cam(a, r, f"{CLASSES[int(r['label'])]} {CLASS_NAMES[CLASSES[int(r['label'])]]} ({r['conf']:.2f})")
plt.suptitle(f"Grad-CAM on correct test predictions - {BEST}")
plt.tight_layout(); plt.savefig(FIG / "gradcam_correct.png"); plt.show()
cam.remove()

**Misclassified examples — likely reasons** (from the run reported in `outputs/`):

1. **phone left → texting left (0.83):** the phone is being lowered from the ear, so for one frame it is held in front of the chest like a texting phone.
2. **talking to passenger → safe driving (0.82):** both hands are on the wheel; the only cue is the head turned to the right. Grad-CAM looks at the arms and wheel, not the face.
3. **drinking → hair & makeup (0.79):** the cup is at the mouth with the elbow raised, the same arm pose as touching the hair.
4. **safe driving → talking to passenger (0.78):** the head is turned slightly towards the passenger while both hands stay on the wheel.
5. **hair & makeup → safe driving (0.72)** and **6. radio → safe driving (0.70):** the hand is already back near the wheel, between two actions.
7. **talking to passenger → radio (0.56):** the right hand gestures low near the centre console, where the radio is.
8. **hair & makeup → phone right (0.56):** a hand held on the cheek looks like a phone held to the ear.

Common cause: a single frame cannot show whether the hand is moving towards or away from the face, and the head direction (the cue for c9) is a small part of the image.

## 10. Robustness test (innovation)
A deployed in-car camera sees conditions that the training set does not: night driving, blur from vibration or a dirty lens, and low-resolution cheap cameras. The selected model is evaluated on the same test images after applying each corruption (no retraining).

In [ ]:
CONDITIONS = {
    "clean": [],
    "low light (brightness x0.5)": [transforms.Lambda(lambda im: TF.adjust_brightness(im, 0.5))],
    "blur (Gaussian, sigma 2)": [transforms.GaussianBlur(9, sigma=2.0)],
    "low resolution (80x60 upscaled)": [transforms.Lambda(lambda im: im.resize((80, 60)).resize(im.size))],
}
rob_rows = []
for cond, extra in CONDITIONS.items():
    tfm = transforms.Compose(extra + list(eval_tfms.transforms))
    dl = DataLoader(DriverDataset(test_df, tfm), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    p, y = predict(best_model, dl)
    m = compute_metrics(y, p)
    rob_rows.append({"condition": cond, "test accuracy": m["accuracy"], "test macro F1": m["macro_f1"]})
robust = pd.DataFrame(rob_rows).set_index("condition")
robust["drop in macro F1"] = robust.loc["clean", "test macro F1"] - robust["test macro F1"]
robust.round(3)

## 11. Specific challenge: random split vs driver-grouped split
The selected architecture is trained again with **exactly the same recipe and subset**, but on the random split from Section 2. The only difference is whether test drivers were seen in training.

In [ ]:
loaders_rand = make_loaders(rand)
leak_model, leak_hist, leak_info = fit(BEST, loaders_rand, tag="random")
plot_history(leak_hist, f"{BEST} - RANDOM split (drivers shared between train and test)",
             FIG / f"curves_{BEST}_random_split.png")
p, y = predict(leak_model, loaders_rand["test"])
leak_test = compute_metrics(y, p)
drv_test = compute_metrics(y_test, p_best)

leak_table = pd.DataFrame({
    "random split (drivers shared)": {"val macro F1": leak_info["best_val_f1"], "test accuracy": leak_test["accuracy"],
                                      "test macro F1": leak_test["macro_f1"], "test log loss": leak_test["log_loss"]},
    "driver-grouped split (unseen drivers)": {"val macro F1": runs[BEST]["info"]["best_val_f1"],
                                              "test accuracy": drv_test["accuracy"],
                                              "test macro F1": drv_test["macro_f1"], "test log loss": drv_test["log_loss"]},
}).T
leak_table.to_csv(OUT / "leakage_experiment.csv")

ax = leak_table[["test accuracy", "test macro F1"]].plot.bar(figsize=(6.5, 3.4), rot=0, color=["#4C72B0", "#55A868"])
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title(f"{BEST}: same model, different split"); ax.set_ylabel("score on test split")
plt.savefig(FIG / "leakage_bar.png"); plt.show()
print(f"a random split overstates test accuracy by "
      f"{(leak_test['accuracy'] - drv_test['accuracy']) * 100:.1f} percentage points")
leak_table.round(3)

## 12. Summary
All numbers used in the report are collected below and saved to `outputs/results.json`. Every number in the report comes from the **test drivers** unless it says otherwise.

In [ ]:
results = {
    "dataset": {"total_labelled": int(len(df)), "drivers": int(df["subject"].nunique()),
                "subset_images": int(len(sub)), "per_driver_class": PER_DRIVER_CLASS,
                "class_counts_full": cc.to_dict()},
    "split": {s: {"drivers": d, "images": int((sub["split"] == s).sum())}
              for s, d in [("train", train_drivers), ("val", val_drivers), ("test", test_drivers)]},
    "near_duplicates": nn_stats,
    "validation_table": val_table.round(4).to_dict(orient="index"),
    "selected_model": BEST,
    "test_table": test_table.round(4).to_dict(orient="index"),
    "best_per_class": prf.round(4).to_dict(orient="index"),
    "most_confused": [{"true": a, "pred": b, "count": int(n)} for n, a, b in confused],
    "per_driver_accuracy": per_driver.round(4).to_dict(),
    "robustness": robust.round(4).to_dict(orient="index"),
    "leakage_experiment": leak_table.round(4).to_dict(orient="index"),
    "hyperparameters": {"img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs_scratch": EPOCHS_SCRATCH,
                        "epochs_head": EPOCHS_HEAD, "epochs_finetune": EPOCHS_FT, "patience": PATIENCE,
                        "lr_scratch": LR_SCRATCH, "lr_head": LR_HEAD, "lr_finetune": LR_FT,
                        "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
                        "optimizer": "AdamW + OneCycleLR", "seed": SEED},
    "versions": versions,
}
(OUT / "results.json").write_text(json.dumps(results, indent=2, default=float))

b = test_table.loc[BEST]
print(f"Selected model : {BEST} (chosen on validation macro F1)")
print(f"Test drivers   : {', '.join(test_drivers)} ({(sub['split'] == 'test').sum()} images)")
print(f"Test accuracy  : {b['test accuracy']:.2f}   macro F1: {b['test macro F1']:.2f}   "
      f"log loss: {b['test log loss']:.2f}   top-2: {b['test top-2 acc']:.2f}")
print(f"Random split would report accuracy {leak_test['accuracy']:.2f} on the same data -> leakage")

In [ ]:
# Colab: zip the figures, splits, tables and results.json (model weights left out) and download them for the report
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("PS48_outputs", "zip", ".", "outputs")
    subprocess.run(["zip", "-q", "-d", "PS48_outputs.zip", "outputs/models/*"])
    files.download("PS48_outputs.zip")

**Analysis and conclusions** (numbers from the run in `outputs/results.json`)

* **Result:** ResNet-18 was selected on validation macro F1 (0.85). On the 5 unseen test drivers it reaches 89.75 % accuracy, macro F1 0.90, top-2 accuracy 96.00 % and log loss 0.48.
* **Confused classes:** talking to passenger → safe driving (27 of 80) is by far the largest error; both poses have both hands on the wheel. Safe driving therefore has the lowest precision (0.64). Texting right → texting left (7) is the next pair.
* **Over-fitting:** training accuracy reaches 99.9 % while validation accuracy on unseen drivers levels off at about 84 %. The gap is the difference between drivers, not memorised images; early stopping kept epoch 11.
* **Effect of the driver-grouped split:** the same model on a random split scores 98.75 % (log loss 0.14), 9 points higher. 50.6 % of random-split test images have a near-copy in training, against 0 % with the driver split.
* **Robustness:** macro F1 drops by 0.01 in low light, 0.07 at low resolution and 0.10 under blur.
* **Next improvement:** use a short sequence of frames or a crop of the head, so that head direction (talking to passenger) can be seen.

**References.** State Farm Distracted Driver Detection, Kaggle competition, https://www.kaggle.com/c/state-farm-distracted-driver-detection. Pretrained weights: torchvision (ImageNet-1k). Grad-CAM: Selvaraju et al., ICCV 2017.